In [ ]:
# Import libraries

import os
import json
import time
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader as TorchDataLoader

import monai
from monai.data import Dataset, DataLoader, decollate_batch
from monai.networks.nets import UNet
from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.transforms import (
    LoadImaged,
    EnsureChannelFirstd,
    ConcatItemsd,
    NormalizeIntensityd,
    RandCropByPosNegLabeld,
    AsDiscrete,
    Compose,
)
from monai.inferers import sliding_window_inference

import wandb
from dotenv import load_dotenv

load_dotenv()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"MONAI version: {monai.__version__}")
print(f"Using device: {device}")

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\brain-tumor-segmentation-3d\repo"

PATHS = {
    "training_dir": os.path.join(BASE_DIR, "data", "brats2020", "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData"),
    "configs": os.path.join(BASE_DIR, "configs"),
    "models": os.path.join(BASE_DIR, "models", "baseline"),
    "figures": os.path.join(BASE_DIR, "results", "figures"),
    "metrics": os.path.join(BASE_DIR, "results", "metrics"),
}

for key in ["configs", "models", "figures", "metrics"]:
    os.makedirs(PATHS[key], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:14s} -> {path}")

In [ ]:
# Build the 3D U-Net model

model = UNet(
    spatial_dims=3,
    in_channels=4,       # 4 MRI modalities (T1, T1ce, T2, FLAIR)
    out_channels=4,      # 4 classes (background, NCR/NET, ED, ET)
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model built: 3D U-Net")
print(f"Total parameters: {n_params:,}")

In [ ]:
# Define loss function and label remapping
# BraTS labels are {0, 1, 2, 4} — we remap 4 -> 3 so classes are contiguous (0,1,2,3)

from monai.transforms import MapTransform

class RemapLabeld(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            d[key][d[key] == 4] = 3
        return d


loss_function = DiceFocalLoss(
    to_onehot_y=True,
    softmax=True,
    lambda_dice=0.5,
    lambda_focal=0.5,
)

print("Loss function: DiceFocalLoss (0.5 * Dice + 0.5 * Focal)")
print("Label remapping: 4 -> 3 (so classes are contiguous: 0=background, 1=NCR/NET, 2=ED, 3=ET)")

In [ ]:
# Load the dataset split saved in notebook 02

split_path = os.path.join(PATHS["configs"], "dataset_split.json")
with open(split_path, "r") as f:
    split_dict = json.load(f)

train_patients = split_dict["train"]
val_patients = split_dict["val"]

modalities = ["t1", "t1ce", "t2", "flair"]

def build_data_dicts(patient_list, training_dir):
    data_dicts = []
    for pid in patient_list:
        patient_dir = os.path.join(training_dir, pid)
        entry = {mod: os.path.join(patient_dir, f"{pid}_{mod}.nii") for mod in modalities}
        entry["label"] = os.path.join(patient_dir, f"{pid}_seg.nii")
        entry["patient_id"] = pid
        data_dicts.append(entry)
    return data_dicts

train_dicts = build_data_dicts(train_patients, PATHS["training_dir"])
val_dicts = build_data_dicts(val_patients, PATHS["training_dir"])

patch_size = (96, 96, 96)

# Training transforms: load -> remap labels -> fuse modalities -> normalize -> extract patches
train_transforms = Compose([
    LoadImaged(keys=modalities + ["label"]),
    EnsureChannelFirstd(keys=modalities + ["label"]),
    RemapLabeld(keys=["label"]),
    ConcatItemsd(keys=modalities, name="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=patch_size,
        pos=1,
        neg=1,
        num_samples=2,
        image_key="image",
        image_threshold=0,
    ),
])

# Validation transforms: same, but no random patch cropping (we evaluate on full volumes)
val_transforms = Compose([
    LoadImaged(keys=modalities + ["label"]),
    EnsureChannelFirstd(keys=modalities + ["label"]),
    RemapLabeld(keys=["label"]),
    ConcatItemsd(keys=modalities, name="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

train_ds = Dataset(data=train_dicts, transform=train_transforms)
val_ds = Dataset(data=val_dicts, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=0)

print(f"Train dataset: {len(train_ds)} patients")
print(f"Val dataset: {len(val_ds)} patients")
print(f"Patch size: {patch_size}, patches per volume: 2")

In [ ]:
# Training loop setup

wandb_key = os.environ.get("WANDB_API_KEY")
if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
    print("W&B login successful")
else:
    print("WARNING: WANDB_API_KEY not found in environment")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

num_epochs = 15  # baseline run; we'll revisit this after seeing initial results
val_interval = 5

dice_metric = DiceMetric(include_background=False, reduction="mean")
post_pred = AsDiscrete(argmax=True, to_onehot=4)
post_label = AsDiscrete(to_onehot=4)

wandb.init(
    project="brain-tumor-segmentation-3d",
    name="baseline-3dunet",
    config={
        "architecture": "3D U-Net (MONAI)",
        "patch_size": patch_size,
        "batch_size": 1,
        "patches_per_volume": 2,
        "lr": 1e-4,
        "num_epochs": num_epochs,
        "loss": "DiceFocalLoss",
    },
)

best_dice = -1
best_epoch = -1

print(f"Starting training: {num_epochs} epochs, validating every {val_interval} epochs")
print(f"Train batches per epoch: {len(train_loader)}")

In [ ]:
# Main training loop

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    start_time = time.time()

    for batch_data in train_loader:
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    epoch_time = time.time() - start_time

    wandb.log({"train_loss": epoch_loss, "epoch": epoch, "epoch_time_sec": epoch_time})
    print(f"Epoch {epoch+1}/{num_epochs} — loss: {epoch_loss:.4f} — time: {epoch_time:.1f}s")

    # Validation
    if (epoch + 1) % val_interval == 0:
        model.eval()
        dice_metric.reset()

        with torch.no_grad():
            for val_data in val_loader:
                val_inputs = val_data["image"].to(device)
                val_labels = val_data["label"].to(device)

                val_outputs = sliding_window_inference(
                    val_inputs, patch_size, sw_batch_size=1, predictor=model
                )

                val_outputs_list = decollate_batch(val_outputs)
                val_labels_list = decollate_batch(val_labels)

                val_outputs_convert = [post_pred(x) for x in val_outputs_list]
                val_labels_convert = [post_label(x) for x in val_labels_list]

                dice_metric(y_pred=val_outputs_convert, y=val_labels_convert)

        mean_dice = dice_metric.aggregate().item()
        wandb.log({"val_dice": mean_dice, "epoch": epoch})
        print(f"  -> Validation Dice: {mean_dice:.4f}")

        if mean_dice > best_dice:
            best_dice = mean_dice
            best_epoch = epoch
            torch.save(model.state_dict(), os.path.join(PATHS["models"], "best_model.pth"))
            print(f"  -> New best model saved (Dice: {best_dice:.4f})")

print(f"\nTraining complete. Best Dice: {best_dice:.4f} at epoch {best_epoch+1}")

In [ ]:
# Load the best model and visualize predictions on a validation patient

model.load_state_dict(torch.load(os.path.join(PATHS["models"], "best_model.pth")))
model.eval()

sample_val = val_dicts[0]
sample_transformed = val_transforms(sample_val)

input_volume = sample_transformed["image"].unsqueeze(0).to(device)
label_volume = sample_transformed["label"]

with torch.no_grad():
    pred_volume = sliding_window_inference(input_volume, patch_size, sw_batch_size=1, predictor=model)
    pred_volume = torch.argmax(pred_volume, dim=1).squeeze(0).cpu()

label_np = label_volume.squeeze(0).numpy()
pred_np = pred_volume.numpy()

# Find the slice with the most ground-truth tumor for a meaningful comparison
tumor_per_slice = (label_np > 0).sum(axis=(0, 1))
best_slice = np.argmax(tumor_per_slice)

flair_np = sample_transformed["image"][3].numpy()  # channel 3 = FLAIR

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(flair_np[:, :, best_slice].T, cmap="gray", origin="lower")
axes[0].set_title("FLAIR")
axes[0].axis("off")

axes[1].imshow(flair_np[:, :, best_slice].T, cmap="gray", origin="lower")
gt_mask = np.ma.masked_where(label_np[:, :, best_slice].T == 0, label_np[:, :, best_slice].T)
axes[1].imshow(gt_mask, cmap="autumn", alpha=0.6, origin="lower")
axes[1].set_title("Ground Truth")
axes[1].axis("off")

axes[2].imshow(flair_np[:, :, best_slice].T, cmap="gray", origin="lower")
pred_mask = np.ma.masked_where(pred_np[:, :, best_slice].T == 0, pred_np[:, :, best_slice].T)
axes[2].imshow(pred_mask, cmap="autumn", alpha=0.6, origin="lower")
axes[2].set_title("Model Prediction")
axes[2].axis("off")

fig.suptitle(f"{sample_val['patient_id']} — slice {best_slice}", fontsize=13)
plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "baseline_prediction_sample.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Summary

print("NOTEBOOK 03 COMPLETE")
print("=" * 50)
print(f"Model: 3D U-Net (MONAI), {n_params:,} parameters")
print(f"Loss: DiceFocalLoss (0.5 * Dice + 0.5 * Focal)")
print(f"Training: {num_epochs} epochs, patch size {patch_size}")
print()
print(f"Validation Dice progression:")
print(f"  Epoch 5:  0.5792")
print(f"  Epoch 10: 0.6120")
print(f"  Epoch 15: 0.6570 (best)")
print(f"  -> Dice was still improving at the final epoch — more epochs likely to help further")
print()
print(f"Model saved -> models/baseline/best_model.pth")
print(f"W&B run: https://wandb.ai/foroughm423/brain-tumor-segmentation-3d")
print()
print("Figures saved:")
for fname in ["baseline_prediction_sample.png"]:
    path = os.path.join(PATHS["figures"], fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")
print()
print("Next -> 04_error_analysis.ipynb")
print("  - Systematically check where the model over/under-segments")
print("  - Look for patterns: tumor size, specific classes (NCR/ED/ET), location")
print("  - Decide evidence-based next steps (more epochs, architecture change, etc.)")